# Modal Analysis — Analysis Methods

This notebook documents the signal-processing methods used in Pages 1–3 of the Streamlit app.

**System model throughout:**

$$x(t) \xrightarrow{h(t)} y(t)$$

where $x$ is the input (force/excitation) and $y$ is the output (response).

---
## Page 1 — Time History

### 1.1 Time-range trimming

A boolean mask selects the samples within the user-chosen window $[t_1, t_2]$:

$$\text{mask} = \{i : t_1 \leq t_i \leq t_2\}$$

All downstream statistics and spectral pages operate on the trimmed signal.

### 1.2 Butterworth digital filter

The filter is designed with `scipy.signal.butter` and applied zero-phase with `sosfiltfilt`.

#### Design

An $N$-th order Butterworth filter has a maximally-flat magnitude response:

$$|H(f)|^2 = \frac{1}{1 + \left(\dfrac{f}{f_c}\right)^{2N}}$$

The normalised cutoff frequency passed to scipy is:

$$W_n = \frac{f_c}{f_{\text{Nyquist}}} = \frac{f_c}{f_s/2} \in (0, 1)$$

For two-sided filters (bandpass / bandstop) $W_n = [W_{n,\text{low}},\; W_{n,\text{high}}]$.

| Filter type | `btype` | Passband |
|---|---|---|
| Lowpass | `'low'` | $f \leq f_c$ |
| Highpass | `'high'` | $f \geq f_c$ |
| Bandpass | `'bandpass'` | $f_{\text{low}} \leq f \leq f_{\text{high}}$ |
| Bandstop | `'bandstop'` | $f \leq f_{\text{low}}$ or $f \geq f_{\text{high}}$ |

#### Zero-phase application

`sosfiltfilt` performs two passes (forward and reverse), yielding zero group delay and doubling the effective filter order:

$$y = \text{sosfiltfilt}(\text{sos},\; x) \implies |H_{\text{eff}}(f)|^2 = |H(f)|^4$$

The output has the same length as the input; edge effects are minimised by padding.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import butter, sosfiltfilt, freqz_sos

fs = 1000.0          # example sample rate (Hz)
nyq = fs / 2.0
order = 4
fc = 100.0           # cutoff (Hz)

sos_lp = butter(order, fc / nyq, btype='low', output='sos')
sos_hp = butter(order, fc / nyq, btype='high', output='sos')
sos_bp = butter(order, [50/nyq, 200/nyq], btype='bandpass', output='sos')
sos_bs = butter(order, [50/nyq, 200/nyq], btype='bandstop', output='sos')

fig, axes = plt.subplots(2, 2, figsize=(12, 7), sharex=True)
labels = ['Lowpass 100 Hz', 'Highpass 100 Hz', 'Bandpass 50–200 Hz', 'Bandstop 50–200 Hz']
for ax, sos, label in zip(axes.flat, [sos_lp, sos_hp, sos_bp, sos_bs], labels):
    w, h = freqz_sos(sos, worN=2048, fs=fs)
    ax.plot(w, 20 * np.log10(np.maximum(np.abs(h), 1e-12)))
    ax.set_title(f'Order {order} Butterworth — {label}')
    ax.set_xlabel('Frequency (Hz)')
    ax.set_ylabel('Magnitude (dB)')
    ax.set_ylim(-80, 5)
    ax.grid(True)
plt.tight_layout()
plt.show()

---
## Page 2 — FFT

Page 2 offers two methods: **Single FFT** and **Welch PSD**.

### 2.1 Single FFT

#### Windowing

Before transforming, each signal is multiplied by a window $w[n]$ of the same length $N$:

$$\tilde{x}[n] = x[n] \cdot w[n], \quad n = 0, 1, \ldots, N-1$$

Windows available and their typical uses:

| App label | scipy name | Best for |
|---|---|---|
| Uniform | `boxcar` | Periodic signals, no spectral leakage concern |
| Hanning | `hann` | General-purpose, good sidelobe rejection |
| Flat Top | `flattop` | Accurate amplitude measurement |
| Force | `hann` | Impulsive input (short Hann window) |
| Exponential | `exponential` | Decaying free-vibration responses |

#### One-sided DFT

The one-sided (real-input) DFT via `np.fft.rfft`:

$$X[k] = \sum_{n=0}^{N-1} \tilde{x}[n]\, e^{-j 2\pi k n / N}, \quad k = 0, 1, \ldots, \lfloor N/2 \rfloor$$

Frequency bins (Hz):

$$f_k = \frac{k \cdot f_s}{N}, \quad \Delta f = \frac{f_s}{N}$$

#### Display modes

**Gain / Phase:**
$$\text{Gain}[k] = |X[k]|, \qquad \text{Phase}[k] = \angle X[k] \text{ (deg)}$$

**Real / Imaginary:**
$$\text{Real}[k] = \operatorname{Re}\{X[k]\}, \qquad \text{Imag}[k] = \operatorname{Im}\{X[k]\}$$

### 2.2 Welch PSD

Welch's method reduces variance by averaging periodograms over $K$ overlapping segments of length $L$ (`nperseg`).

$$\hat{S}_{xx}(f) = \frac{1}{K} \sum_{k=0}^{K-1} \frac{1}{f_s \, U} \left| \sum_{n=0}^{L-1} x_k[n]\, w[n]\, e^{-j2\pi f n / f_s} \right|^2$$

where $U = \frac{1}{L}\sum_{n=0}^{L-1} w[n]^2$ normalises for the window power.

Key parameters:

| Parameter | Symbol | Effect |
|---|---|---|
| Segment length | $L$ = `nperseg` $= N / K$ | Frequency resolution $\Delta f = f_s / L$ |
| Overlap | `noverlap` $= L \times p / 100$ | More overlap → more segments → lower variance |
| Window | `window` | Trades sidelobe rejection vs. amplitude accuracy |

In [ ]:
from scipy.signal import get_window, welch as scipy_welch

# --- Window shapes ---
N = 512
windows = {
    'boxcar (Uniform)': 'boxcar',
    'hann (Hanning/Force)': 'hann',
    'flattop (Flat Top)': 'flattop',
    'exponential': ('exponential', None, 1.0 / 8.686),
}

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

for name, spec in windows.items():
    w = get_window(spec, N)
    axes[0].plot(w, label=name)
    W = np.fft.rfft(w, n=4096)
    f = np.fft.rfftfreq(4096)
    axes[1].plot(f, 20 * np.log10(np.maximum(np.abs(W) / np.max(np.abs(W)), 1e-6)), label=name)

axes[0].set_title('Window functions (N=512)')
axes[0].set_xlabel('Sample')
axes[0].set_ylabel('Amplitude')
axes[0].legend(fontsize=8)
axes[0].grid(True)

axes[1].set_title('Window spectra (normalised)')
axes[1].set_xlabel('Normalised frequency')
axes[1].set_ylabel('Magnitude (dB)')
axes[1].set_ylim(-120, 5)
axes[1].legend(fontsize=8)
axes[1].grid(True)

plt.tight_layout()
plt.show()

In [ ]:
# --- Welch variance reduction demo ---
rng = np.random.default_rng(0)
fs = 1000.0
t = np.arange(0, 10, 1/fs)
# 50 Hz sinusoid + broadband noise
x = np.sin(2 * np.pi * 50 * t) + 0.5 * rng.standard_normal(len(t))

fig, ax = plt.subplots(figsize=(12, 4))

for n_segs, style in [(1, 'dotted'), (4, 'dashed'), (16, 'solid')]:
    nperseg = max(16, len(x) // n_segs)
    f, Pxx = scipy_welch(x, fs=fs, window='hann', nperseg=nperseg, noverlap=nperseg//2)
    ax.semilogy(f, Pxx, linestyle=style, label=f'{n_segs} segment(s), Δf={fs/nperseg:.2f} Hz')

ax.set_title('Welch PSD — effect of number of segments')
ax.set_xlabel('Frequency (Hz)')
ax.set_ylabel('PSD (unit²/Hz)')
ax.legend()
ax.grid(True)
plt.tight_layout()
plt.show()

---
## Page 3 — Spectral Analysis

### 3.1 Spectral quantities

Given complex FFT arrays $S_x[k]$ (input) and $S_y[k]$ (output), or their Welch-averaged equivalents:

| Quantity | Formula | Units |
|---|---|---|
| Input auto-power | $G_{xx} = S_x S_x^*$ | amplitude² |
| Output auto-power | $G_{yy} = S_y S_y^*$ | amplitude² |
| Cross-power (yx) | $G_{yx} = S_y S_x^*$ | complex amplitude² |
| Cross-power (xy) | $G_{xy} = G_{yx}^* = S_x S_y^*$ | complex amplitude² |

For Welch, these are ensemble-averaged over $K$ segments before the FRF and coherence are formed.

### 3.2 PSD normalisation

The auto-power from `scipy.signal.welch` is already a two-sided PSD normalised to unit²/Hz.  
For the **Single FFT** path the raw periodogram is converted:

$$S_{xx}(f) = \frac{2\, G_{xx}[k]}{f_s \, N}$$

(factor of 2 folds the negative-frequency energy into the one-sided spectrum).

### 3.3 FRF estimators

Three estimators handle different noise scenarios:

$$H_1(f) = \frac{G_{yx}}{G_{xx}} \qquad \text{(minimises output noise bias)}$$

$$H_2(f) = \frac{G_{yy}}{G_{xy}} \qquad \text{(minimises input noise bias)}$$

$$H_v(f) = \sqrt{|H_1||H_2|}\, e^{j\angle H_1} \qquad \text{(geometric-mean magnitude, } H_1 \text{ phase)}$$

All three are identical when coherence $\gamma^2 = 1$.

### 3.4 Ordinary coherence

$$\gamma^2(f) = \frac{|G_{yx}|^2}{G_{xx}\, G_{yy}} \in [0, 1]$$

- $\gamma^2 = 1$ for a single-realisation FFT (always, by definition).
- For Welch estimates, $\gamma^2 < 1$ indicates extraneous noise, nonlinearity, or leakage.
- The app flags $\gamma^2 = 0.85$ as a practical acceptability threshold.

### 3.5 Method comparison

| Property | Single FFT | Welch |
|---|---|---|
| Frequency resolution $\Delta f$ | $f_s / N$ (maximum) | $f_s / L$, reduced |
| Variance | High | Reduced by $\sim 1/K$ |
| Coherence | Always 1 | Meaningful |
| Input required | Saved `fft_results` | Raw / processed signals |

In [ ]:
import sys, os
# Add project root so core.spectral is importable
sys.path.insert(0, os.path.join(os.getcwd()))

from core.spectral import compute_fft, compute_spectral_quantities, compute_welch_quantities

# --- Synthetic SDOF system ---
rng = np.random.default_rng(42)
fs = 1000.0
t = np.arange(0, 20, 1/fs)
N = len(t)

# White noise input
x = rng.standard_normal(N)

# SDOF impulse response: fn=50 Hz, zeta=0.05
fn, zeta = 50.0, 0.05
wn = 2 * np.pi * fn
wd = wn * np.sqrt(1 - zeta**2)
dt = 1.0 / fs
h = np.exp(-zeta * wn * np.arange(N) * dt) * np.sin(wd * np.arange(N) * dt) / wd

# Response = convolution + measurement noise
y_clean = np.convolve(h, x)[:N]
y = y_clean + 0.05 * rng.standard_normal(N)

# --- Single FFT ---
freqs_s, Sx = compute_fft(x, fs, window='hanning')
_, Sy = compute_fft(y, fs, window='hanning')
sq_s = compute_spectral_quantities(Sx, Sy)

# --- Welch ---
sq_w = compute_welch_quantities(x, y, fs, nperseg=1024, noverlap=512, window='hann')
freqs_w = sq_w['freqs']

# --- Plot ---
eps = np.finfo(float).tiny

fig, axes = plt.subplots(3, 2, figsize=(14, 10), sharex=True)

for col, (method_label, freqs, sq) in enumerate([
    ('Single FFT', freqs_s, sq_s),
    ('Welch (1024 samples/seg, 50% overlap)', freqs_w, sq_w),
]):
    mask = freqs <= 200
    f = freqs[mask]

    # Auto-power (dB)
    axes[0, col].plot(f, 10 * np.log10(np.maximum(sq['Gxx'][mask], eps)), label='Gxx (input)')
    axes[0, col].plot(f, 10 * np.log10(np.maximum(sq['Gyy'][mask], eps)), label='Gyy (output)', alpha=0.8)
    axes[0, col].set_title(f'Auto-power — {method_label}')
    axes[0, col].set_ylabel('PSD (dB)')
    axes[0, col].legend(fontsize=8)
    axes[0, col].grid(True)

    # FRF magnitude (dB)
    H1 = sq['H1'][mask]
    H2 = sq['H2'][mask]
    Hv = sq['Hv'][mask]
    axes[1, col].plot(f, 20 * np.log10(np.maximum(np.abs(H1), eps)), label='H1', lw=1.2)
    axes[1, col].plot(f, 20 * np.log10(np.maximum(np.abs(H2), eps)), label='H2', lw=1.2, ls='--')
    axes[1, col].plot(f, 20 * np.log10(np.maximum(np.abs(Hv), eps)), label='Hv', lw=1.5, ls=':')
    axes[1, col].set_title(f'FRF — {method_label}')
    axes[1, col].set_ylabel('|H| (dB)')
    axes[1, col].legend(fontsize=8)
    axes[1, col].grid(True)

    # Coherence
    axes[2, col].plot(f, sq['gamma2'][mask])
    axes[2, col].axhline(0.85, color='gray', ls='--', lw=1, label='γ²=0.85')
    axes[2, col].set_title(f'Coherence — {method_label}')
    axes[2, col].set_xlabel('Frequency (Hz)')
    axes[2, col].set_ylabel('γ²')
    axes[2, col].set_ylim(0, 1.05)
    axes[2, col].legend(fontsize=8)
    axes[2, col].grid(True)

plt.suptitle('Single FFT vs. Welch — synthetic SDOF (fn=50 Hz, ζ=0.05)', y=1.01, fontsize=12)
plt.tight_layout()
plt.show()

---
## Summary

| Page | Core operation | Key scipy functions |
|---|---|---|
| 1 — Time History | Zero-phase Butterworth filtering + time trim | `butter`, `sosfiltfilt` |
| 2 — FFT | Windowed DFT **or** Welch PSD per channel | `get_window`, `np.fft.rfft`, `welch` |
| 3 — Spectral Analysis | Auto/cross-power, H1/H2/Hv FRFs, coherence | `welch`, `csd` (via `core.spectral`) |